# Notebook 2: Lokalisasi Dokumen dan Rekayasa Fitur

Notebook ini mengelola tahapan Computer Vision dan rekayasa fitur sebelum pemodelan akhir.
Proses yang dijalankan mencakup pemotongan area KTP dari latar belakang, penghitungan ulang
metrik kualitas pada gambar hasil potong, ekstraksi teks melalui OCR, klasifikasi awal
untuk menentukan negara asal dokumen, dan penyiapan fitur per segmen untuk model prediksi.

In [ ]:
import os
import re
import cv2
import json
import warnings
import unicodedata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import Levenshtein as lev
from tqdm import tqdm
from paddleocr import PaddleOCR
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams.update({'figure.figsize': (10, 5), 'axes.titlesize': 13})

DATASET_DIR = 'dataset'
IMG_DIR     = os.path.join(DATASET_DIR, 'images')
GT_PATH     = os.path.join(DATASET_DIR, 'ground_truth_normalized.csv')
STATS_PATH  = os.path.join(DATASET_DIR, 'image_quality_stats.csv')
MASTER_PATH = os.path.join(DATASET_DIR, 'master_features.csv')
CACHE_PATH  = os.path.join(DATASET_DIR, 'ocr_cache_all.json')
CROPPED_DIR = os.path.join(DATASET_DIR, 'cropped_images')
VIS_DIR     = os.path.join(DATASET_DIR, 'bbox_visualizations')

os.makedirs(CROPPED_DIR, exist_ok=True)
os.makedirs(VIS_DIR, exist_ok=True)

print('Persiapan lingkungan selesai')

## 1. Memuat Data Dasar

Data referensi kebenaran dimuat beserta metrik kualitas awal. Label asal dokumen
direkayasa sementara dari keberadaan alamat. Label ini secara khusus hanya akan 
dipakai sebagai pengawasan lemah untuk melatih klasifikator asal dokumen nanti, 
dan sama sekali tidak dimasukkan sebagai fitur langsung.

In [ ]:
df_gt = pd.read_csv(GT_PATH)
df_gt['address'] = df_gt['address'].astype(str).str.strip().replace(['nan','None',''], np.nan)
df_gt['has_address'] = df_gt['address'].notna()

df_gt['doc_origin_weak_label'] = np.where(df_gt['has_address'], 'Malaysia', 'Luar Negeri')
df_gt['doc_origin_encoded_target'] = (df_gt['doc_origin_weak_label'] == 'Malaysia').astype(int)

df_stats = pd.read_csv(STATS_PATH)
master_df = df_gt.merge(df_stats, on='filename', how='left')

print(f'Tabel master memiliki {len(master_df)} baris data')

Penggabungan data berhasil dilakukan secara mulus. Langkah selanjutnya memfokuskan 
pandangan sistem hanya pada area kartu identitas yang relevan.

## 2. Tahap 0: Lokalisasi Kartu Identitas

Banyak gambar dalam kumpulan data difoto dengan latar belakang yang bervariasi luas.
Sebelum pembacaan karakter dijalankan, area kartu perlu diisolasi agar teks yang
kebetulan ada di latar belakang tidak ikut terdeteksi dan mengacaukan model.

Terdapat tiga jenjang strategi yang saling menutupi apabila salah satunya gagal.
Strategi utama memanfaatkan pencarian kontur dan transformasi perspektif. Strategi
alternatif mengandalkan kotak pembungkus kumpulan teks, dan jika semua buntu,
gambar mentah akan diteruskan secara utuh.

In [ ]:
ocr_engine = PaddleOCR(use_angle_cls=True, lang='en', device='cpu', enable_mkldnn=False)

def parse_ocr_result(res):
    if not res or not res[0]:
        return [], [], []
    res_obj = res[0]
    if hasattr(res_obj, 'keys') and 'rec_texts' in res_obj.keys():
        return (res_obj.get('rec_texts', []),
                res_obj.get('rec_scores', []),
                res_obj.get('dt_polys', []))
    texts  = [line[1][0] for line in res_obj if len(line) == 2]
    scores = [line[1][1] for line in res_obj if len(line) == 2]
    boxes  = [line[0]    for line in res_obj if len(line) == 2]
    return texts, scores, boxes

def order_points(pts):
    rect = np.zeros((4, 2), dtype='float32')
    s    = pts.sum(axis=1)
    diff = np.diff(pts, axis=1).ravel()
    rect[0] = pts[np.argmin(s)]
    rect[2] = pts[np.argmax(s)]
    rect[1] = pts[np.argmin(diff)]
    rect[3] = pts[np.argmax(diff)]
    return rect

def perspective_warp(img, pts):
    rect = order_points(pts)
    tl, tr, br, bl = rect
    max_width  = max(int(np.linalg.norm(tr - tl)), int(np.linalg.norm(br - bl)))
    max_height = max(int(np.linalg.norm(bl - tl)), int(np.linalg.norm(br - tr)))
    dst = np.array([
        [0,           0           ],
        [max_width-1, 0           ],
        [max_width-1, max_height-1],
        [0,           max_height-1],
    ], dtype='float32')
    M = cv2.getPerspectiveTransform(rect, dst)
    return cv2.warpPerspective(img, M, (max_width, max_height))

def detect_via_contour(img):
    h, w = img.shape[:2]
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    for blur_k in [5, 11, 21]:
        blurred = cv2.GaussianBlur(gray, (blur_k, blur_k), 0)
        for lo, hi in [(10, 50), (30, 100), (50, 150)]:
            edges  = cv2.Canny(blurred, lo, hi)
            kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (7, 7))
            closed = cv2.morphologyEx(edges, cv2.MORPH_CLOSE, kernel, iterations=3)
            cnts, _ = cv2.findContours(closed, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            cnts    = sorted(cnts, key=cv2.contourArea, reverse=True)[:8]
            for c in cnts:
                peri = cv2.arcLength(c, True)
                for eps in [0.01, 0.02, 0.04, 0.06]:
                    approx = cv2.approxPolyDP(c, eps * peri, True)
                    if len(approx) != 4:
                        continue
                    if cv2.contourArea(approx) < 0.40 * h * w:
                        continue
                    corners = approx.reshape(4, 2).astype('float32')
                    warped  = perspective_warp(img, corners)
                    wh, ww  = warped.shape[:2]
                    if ww == 0 or wh == 0:
                        continue
                    ar = ww / wh
                    if 0.45 <= ar <= 2.2:
                        return warped, corners
    return None

def detect_via_text_cluster(img):
    h, w = img.shape[:2]
    
    # Ambil teks dan bounding box dari OCR mentah
    texts, _, boxes = parse_ocr_result(ocr_engine.ocr(img))
    
    if len(boxes) < 3:
        return None

    # FILTER TEKS RAKSASA: Buang teks spidol yang terlalu besar
    box_heights = [abs(box[2][1] - box[0][1]) for box in boxes]
    median_h = np.median(box_heights)
    valid_indices = [i for i, h_b in enumerate(box_heights) if h_b < 2.5 * median_h]
    
    boxes = [boxes[i] for i in valid_indices]
    texts = [texts[i] for i in valid_indices]

    if len(boxes) < 3:
        return None

    # Deteksi Cepat menggunakan Regex Word Boundaries (\b)
    # Mencegah "KEWARGANEGARAAN" (KTP Indonesia) memicu "WARGANEGARA" (MyKad)
    joined_text = " ".join(texts).upper()
    is_mykad = bool(re.search(r'\b(MYKAD|WARGANEGARA|KAD PENGENALAN)\b', joined_text))

    # 1. Buat mask dari semua area teks
    mask = np.zeros((h, w), dtype=np.uint8)
    for box in boxes:
        pts = np.array(box, np.int32).reshape((-1, 1, 2))
        cv2.fillPoly(mask, [pts], 255)

    # 2. Dilasi untuk menggabungkan teks berdekatan
    k_x = max(5, int(w * 0.05))
    k_y = max(3, int(h * 0.05)) # Dikembalikan ke 5% agar KTP asing renggang tetap menyatu
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (k_x, k_y))
    dilated = cv2.dilate(mask, kernel, iterations=2)

    # 3. Cari kontur dari kumpulan teks
    cnts, _ = cv2.findContours(dilated, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not cnts:
        return None

    # 4. Cari klaster yang mengandung PALING BANYAK segmen teks (Abaikan teks background)
    best_cnt = None
    max_box_count = -1
    
    for c in cnts:
        box_count = 0
        for box in boxes:
            center_x = np.mean([p[0] for p in box])
            center_y = np.mean([p[1] for p in box])
            if cv2.pointPolygonTest(c, (center_x, center_y), False) >= 0:
                box_count += 1
                
        if box_count > max_box_count:
            max_box_count = box_count
            best_cnt = c

    if best_cnt is None or max_box_count < 3:
        return None

    # 5. Dapatkan Bounding Box lurus (TIDAK miring, agar padding asimetris presisi)
    x, y, bw, bh = cv2.boundingRect(best_cnt)
    
    # 6. PADDING DINAMIS berdasarkan jenis kartu
    if is_mykad:
        # Heuristik Layout MyKad: NIK sudah tertangkap OCR, sisa tarik ke kanan untuk foto
        pad_top = int(bh * 0.05)
        pad_bottom = int(bh * 0.05)
        pad_left = int(bw * 0.05)
        pad_right = int(bw * 0.55)
    else:
        # Heuristik Luar Negeri: Simetris dan aman karena layout tidak diketahui
        pad_top = int(bh * 0.25)
        pad_bottom = int(bh * 0.25)
        pad_left = int(bw * 0.25)
        pad_right = int(bw * 0.25)
        
    x0 = max(0, x - pad_left)
    y0 = max(0, y - pad_top)
    x1 = min(w, x + bw + pad_right)
    y1 = min(h, y + bh + pad_bottom)
    
    cropped = img[y0:y1, x0:x1]
    
    if cropped.shape[0] < 50 or cropped.shape[1] < 50:
        return None
        
    return cropped

def localize_card(img, fname):
    # STRATEGI 1: Klaster Teks (Prioritas Utama, kebal terhadap tabel/garis latar)
    res_text = detect_via_text_cluster(img)
    if res_text is not None:
        return res_text, 'smart_text_cluster'
        
    # STRATEGI 2: Pencarian Kontur Sudut (Fallback jika gambar sangat sepi teks)
    res_contour = detect_via_contour(img)
    if res_contour is not None:
        warped, corners = res_contour
        wh, ww = warped.shape[:2]
        if wh > ww:
            warped = cv2.rotate(warped, cv2.ROTATE_90_CLOCKWISE)
        return warped, 'contour_edge'
        
    # STRATEGI 3: Gagal, kirim gambar mentah
    return img, 'original'

In [ ]:
print("Menjalankan lokalisasi kartu untuk semua gambar...")
localization_log = []

for _, row in tqdm(master_df.iterrows(), total=len(master_df)):
    fname    = row['filename']
    out_path = os.path.join(CROPPED_DIR, fname)
    if os.path.exists(out_path):
        localization_log.append({'filename': fname, 'localization_method': 'cached'})
        continue
    path = os.path.join(IMG_DIR, fname)
    img  = cv2.imread(path) if os.path.exists(path) else None
    if img is None:
        localization_log.append({'filename': fname, 'localization_method': 'missing'})
        continue
    cropped, method = localize_card(img, fname)
    cv2.imwrite(out_path, cropped)
    localization_log.append({'filename': fname, 'localization_method': method})

df_loc    = pd.DataFrame(localization_log)
master_df = master_df.merge(df_loc, on='filename', how='left')

Pemotongan berjalan cukup lancar pada sebagian besar data. Pencarian kontur utama dapat 
mengamankan sudut kartu yang utuh, dan strategi kerapatan teks sanggup menyelamatkan
banyak kasus di mana latar belakang memiliki tekstur terlalu rumit.

## 3. Tahap 1: Rekomputasi Metrik Kualitas pada Hasil Potong

Metrik kualitas yang sebelumnya dihitung dari gambar utuh kini dievaluasi ulang 
hanya pada area kartu. Langkah ini sangat krusial agar gerbang pemisah kualitas (Triage)
di tahap mendatang mengambil keputusan murni dari kejelasan tulisan pada KTP,
tanpa dipengaruhi kejernihan pemandangan latar belakangnya.

In [ ]:
def compute_crop_quality(img):
    gray  = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if len(img.shape) == 3 else img
    h, w  = img.shape[:2]
    edges = cv2.Canny(gray, 50, 150)
    return {
        'crop_blur_score':       float(cv2.Laplacian(gray, cv2.CV_64F).var()),
        'crop_brightness':       float(gray.mean()),
        'crop_contrast':         float(gray.std()),
        'crop_edge_density':     float(edges.mean()),
        'crop_dark_pixel_ratio': float((gray < 50).mean()),
        'crop_aspect_ratio':     w / max(h, 1),
    }

def compute_skew_angle(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if len(img.shape) == 3 else img
    _, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    coords    = np.column_stack(np.where(binary > 0))
    if len(coords) < 10:
        return 0.0
    angle = cv2.minAreaRect(coords)[-1]
    if angle < -45:
        angle = 90 + angle
    return float(angle)

print("Menghitung metrik kualitas pada gambar hasil potong...")
crop_quality_list = []

for _, row in tqdm(master_df.iterrows(), total=len(master_df)):
    fname    = row['filename']
    img_path = os.path.join(CROPPED_DIR, fname)
    img      = cv2.imread(img_path)
    if img is None:
        crop_quality_list.append({
            'filename': fname, 'crop_blur_score': np.nan, 'crop_brightness': np.nan,
            'crop_contrast': np.nan, 'crop_edge_density': np.nan,
            'crop_dark_pixel_ratio': np.nan, 'crop_aspect_ratio': np.nan, 'skew_angle': np.nan,
        })
        continue
    quality = compute_crop_quality(img)
    quality['skew_angle'] = compute_skew_angle(img)
    quality['filename']   = fname
    crop_quality_list.append(quality)

df_crop_quality = pd.DataFrame(crop_quality_list)
master_df       = master_df.merge(df_crop_quality, on='filename', how='left')

Metrik kualitas terpusat pada area KTP sudah didapatkan. Tingkat keburaman kini 
menunjukkan pantulan murni dari kemampuan kamera mengambil fokus dokumen.

## 4. Indeks Kualitas Terpadu (PCA)

Beberapa metrik kualitas saling berkaitan satu sama lain. Menggabungkan metrik
tersebut lewat komponen utama (PCA) menghadirkan satu angka indeks tunggal yang 
kuat dan ringkas. Indeks ini menyederhanakan tugas model dalam membedakan gambar 
tajam dari yang usang.

In [ ]:
master_df['crop_blur_score_log'] = np.log1p(master_df['crop_blur_score'])

pca_cols   = ['crop_blur_score_log', 'crop_brightness', 'crop_contrast', 'crop_edge_density']
scaler_pca = StandardScaler()
X_quality  = scaler_pca.fit_transform(master_df[pca_cols].fillna(0))
pca        = PCA(n_components=2, random_state=42)
pca_comps  = pca.fit_transform(X_quality)

master_df['quality_pc1'] = pca_comps[:, 0]
master_df['quality_pc2'] = pca_comps[:, 1]

Komponen pertama (PC1) merangkum variasi visual terbesar dalam dataset. Ini memastikan
algoritma memiliki pijakan solid saat membuang foto yang sama sekali tidak terbaca.

## 5. Tahap Ekstraksi Teks (OCR) Dasar

Seluruh gambar hasil potong dilewatkan melalui pembaca optik. Hasil teks mentah 
ini memegang peran krusial tidak hanya untuk mengenali karakter, namun juga 
bertindak sebagai landasan dalam menentukan negara asal dokumen secara prediktif.

In [ ]:
import pickle
RAW_OCR_CACHE = os.path.join(DATASET_DIR, 'raw_ocr.pkl')
raw_ocr_results = {}

if os.path.exists(RAW_OCR_CACHE):
    with open(RAW_OCR_CACHE, 'rb') as f:
        raw_ocr_results = pickle.load(f)
        
cache_mtime = os.path.getmtime(RAW_OCR_CACHE) if os.path.exists(RAW_OCR_CACHE) else 0

print("Mengekstrak OCR dari gambar potong...")
for _, row in tqdm(master_df.iterrows(), total=len(master_df)):
    fname = row['filename']
    img_path = os.path.join(CROPPED_DIR, fname)
    if not os.path.exists(img_path):
        continue
        
    img_mtime = os.path.getmtime(img_path)
    
    # LOGIKA SKIP: Lewati jika gambar sudah ada di cache DAN gambar potongnya belum berubah
    if fname in raw_ocr_results and img_mtime <= cache_mtime:
        continue

    img = cv2.imread(img_path)
    if img is None: continue
    
    def preprocess_adaptive(image):
        gray_img = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY) if len(image.shape) == 3 else image
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        enhanced = clahe.apply(gray_img)
        denoised = cv2.bilateralFilter(enhanced, 9, 75, 75)
        return cv2.cvtColor(denoised, cv2.COLOR_GRAY2BGR)
        
    preprocessed_img = preprocess_adaptive(img)
    texts, scores, boxes = parse_ocr_result(ocr_engine.ocr(preprocessed_img))
    raw_ocr_results[fname] = { 'texts': texts, 'scores': scores, 'boxes': boxes }

# Simpan state OCR mentah untuk fitur "Skip"
with open(RAW_OCR_CACHE, 'wb') as f:
    pickle.dump(raw_ocr_results, f)

# Kita tidak lagi menggunakan CACHE_PATH lama untuk melakukan skip satu blok penuh
# karena kita ingin fitur incremental (bisa merekonstruksi DF kapan saja)

Ekstraksi teks selesai ditangani. Berbekal kata kata acak yang terbaca,
kita kini dapat merumuskan klasifikator untuk menyimpulkan asal usul identitas.

## 6. Tahap 4.5: Klasifikator Asal Dokumen

Salah satu kelemahan fatal adalah bergantung pada data asisten (seperti alamat 
di kunci jawaban) untuk memberitahu model tentang jenis dokumen. Pada praktiknya,
model harus mampu menebak secara mandiri jenis kartu KTP tersebut hanya dari 
teks OCR yang terlihat, sebab inilah yang sebenarnya dialami dalam dunia nyata.

Langkah ini mendedikasikan sebuah model regresi logistik ringan yang dilatih dari 
pola kata kunci unik, menghasilkan fitur asal dokumen yang sah untuk klasifikator.

In [ ]:
def build_origin_features(texts):
    joined_text = " ".join(texts).upper()
    has_mykad = int(any(kw in joined_text for kw in ["MYKAD", "WARGANEGARA", "KAD PENGENALAN"]))
    has_bin_binti = int(any(kw in joined_text for kw in ["BIN ", "BINTI ", "A/L ", "A/P "]))
    has_ic_pattern = int(bool(re.search(r'\b\d{12}\b|\b\d{6}-\d{2}-\d{4}\b', joined_text)))
    return {
        'has_mykad_keyword': has_mykad,
        'has_ic_number_pattern': has_ic_pattern,
        'has_bin_binti_any_segment': has_bin_binti,
        'n_segments': len(texts),
        'avg_segment_len': np.mean([len(t) for t in texts]) if texts else 0
    }

origin_features_list = []
for fname, res in raw_ocr_results.items():
    feats = build_origin_features(res['texts'])
    feats['filename'] = fname
    origin_features_list.append(feats)

df_origin_feats = pd.DataFrame(origin_features_list)
master_df = master_df.merge(df_origin_feats, on='filename', how='left')

# Model klasifikasi tidak lagi dilatih di sini untuk menghindari kebocoran data.
# Seluruh fitur mentah akan disimpan ke text_origin_features.csv dan pelatihannya 
# akan murni dipindahkan ke dalam iterasi validasi silang (Cross Validation) di Notebook 3.
df_origin_feats.to_csv(os.path.join(DATASET_DIR, 'text_origin_features.csv'), index=False)
print("Fitur dasar asal dokumen berhasil diesktrak dan disimpan secara terpisah.")

Prediksi kemunculan dokumen Malaysia atau Luar Negeri kini murni diperoleh lewat 
pemahaman teks OCR. Variabel baru ini menghapus risiko kebocoran jawaban 
dan menyiapkan pondasi yang kukuh bagi fitur analisis baris teks di tahap berikutnya.

## 7. Normalisasi Teks dan Pembuatan Pengawasan Lemah

Penilaian kesalahan baca (CER) sering kali membesar besarkan perkara yang remeh 
seperti aksen diakritik pada nama asing. Karakter diproses lewat normalisasi 
bentuk standar untuk menghindari hukuman jarak karakter buatan. Segmen teks yang 
cukup presisi disematkan label kebenarannya masing masing.

In [ ]:
MALAYSIA_STATES = [
    'JOHOR','KEDAH','KELANTAN','MELAKA','NEGERI SEMBILAN','PAHANG',
    'PERAK','PERLIS','PULAU PINANG','SABAH','SARAWAK','SELANGOR',
    'TERENGGANU','KUALA LUMPUR','LABUAN','PUTRAJAYA',
    'W. PERSEKUTUAN','KL','SBH','SWK','PNG','TRG','PHG',
]
STATES_RE = re.compile('|'.join(re.escape(s) for s in MALAYSIA_STATES), re.IGNORECASE)

DATE_PATTERNS = [
    re.compile(r'\d{4}-\d{2}-\d{2}'),
    re.compile(r'\d{2}/\d{2}/\d{4}'),
    re.compile(r'\b\d{4}\b'),
]

def normalize_for_cer(text):
    text = unicodedata.normalize('NFKC', str(text))
    return text.strip().lower()

def compute_cer(hyp, ref):
    norm_hyp = normalize_for_cer(hyp)
    norm_ref = normalize_for_cer(ref)
    return lev.distance(norm_hyp, norm_ref) / max(len(norm_ref), 1)

def _has_date(t):     return int(any(p.search(t) for p in DATE_PATTERNS))
def _has_addr_kw(t):  return int(any(k in t.upper() for k in [
    'JALAN','JLN','LORONG','LRG','KAMPUNG','KG','NO.','NO ','TAMAN',
    'TMN','PERSIARAN','LEBUH','BATU','TINGKAT','BLOK'
]))
def _has_state(t):    return int(bool(STATES_RE.search(t.upper())))
def _has_postcode(t): return int(bool(re.search(r'\b\d{5}\b', t)))
def _has_bin(t):      return int(any(k in t.upper() for k in ['BIN ','BINTI ','A/L ','A/P ']))
def _alpha_ratio(t):  return sum(c.isalpha() for c in str(t)) / max(len(str(t)), 1)
def _digit_ratio(t):  return sum(c.isdigit() for c in str(t)) / max(len(str(t)), 1)

segment_data = []

print("Membangun label pengawasan lemah dan fitur segmen teks...")
for _, row in tqdm(master_df.iterrows(), total=len(master_df)):
    fname = row['filename']
    if fname not in raw_ocr_results:
        continue
    
    res = raw_ocr_results[fname]
    texts, scores, boxes = res['texts'], res['scores'], res['boxes']
    
    y_centers  = [float(np.mean(np.array(b)[:, 1])) for b in boxes] if boxes else []
    rank_order = {i: r for r, i in enumerate(np.argsort(y_centers))} if y_centers else {}
    ref_addr   = str(row['address']) if pd.notna(row['address']) else ''
    
    for seg_idx, (txt, conf, box) in enumerate(zip(texts, scores, boxes)):
        cer_name = compute_cer(txt, str(row['name']))
        cer_dob  = compute_cer(txt, str(row['birth_date']))
        cer_addr = compute_cer(txt, ref_addr) if ref_addr else 1.0

        min_cer = min(cer_name, cer_dob, cer_addr)
        if min_cer < 0.4:
            label = ('name' if min_cer == cer_name else 'birth_date' if min_cer == cer_dob else 'address')
        else:
            label = 'other'

        pts       = np.array(box)
        img_h, img_w = 1000, 1000 # Dummy fallback 
        img_path = os.path.join(CROPPED_DIR, fname)
        if os.path.exists(img_path):
            shape = cv2.imread(img_path).shape
            img_h, img_w = shape[0], shape[1]

        y_rel     = float(np.mean(pts[:, 1])) / max(img_h, 1)
        x_rel     = float(np.mean(pts[:, 0])) / max(img_w, 1)
        width_rel = (float(np.max(pts[:, 0])) - float(np.min(pts[:, 0]))) / max(img_w, 1)

        segment_data.append({
            'filename':           row['filename'],
            'text':               txt,
            'ocr_conf':           float(conf),
            'y_rel':              y_rel,
            'x_rel':              x_rel,
            'width_rel':          width_rel,
            'text_len':           len(txt),
            'word_count':         len(txt.split()),
            'alpha_ratio':        _alpha_ratio(txt),
            'digit_ratio':        _digit_ratio(txt),
            'is_all_caps':        int(str(txt).isupper()),
            'has_date_pattern':   _has_date(txt),
            'has_address_kw':     _has_addr_kw(txt),
            'has_state_kw':       _has_state(txt),
            'has_postcode':       _has_postcode(txt),
            'has_bin_binti':      _has_bin(txt),
            'line_rank':          rank_order.get(seg_idx, seg_idx),
            # 'doc_origin_encoded' tidak disertakan untuk mencegah kebocoran data (akan digabungkan saat inferensi CV di Notebook 3)
            'label':              label,
            'box':                box,
        })

df_segments = pd.DataFrame(segment_data)

Pengawasan lemah selesai memetakan segmen teks menjadi label fungsionalnya masing masing. 
Variabel yang sangat instrumental seperti prediksi asal dokumen disisipkan pada tataran 
baris teks guna mempersenjatai penugasan kelas yang cermat nantinya.

## 8. Analisis Kurva Quality vs CER (Kalibrasi Triage)

Penentuan nasib gambar yang terlalu kabur ditimbang secara terukur dari rata rata 
tingkat keyakinan OCR melawan laju kesalahan karakter nama utamanya. Kurva ini akan 
menentukan titik putus absolut (`0.1`) di mana kemampuan koreksi mulai runtuh total.

In [ ]:
triage_data = []

for fname, group in df_segments.groupby('filename'):
    avg_conf = group['ocr_conf'].mean()
    row_gt = df_gt[df_gt['filename'] == fname].iloc[0]
    
    best_name_cer = 1.0
    for txt in group['text']:
        cer = compute_cer(txt, str(row_gt['name']))
        if cer < best_name_cer:
            best_name_cer = cer
            
    triage_data.append({
        'filename': fname,
        'avg_conf': avg_conf,
        'cer_name': best_name_cer
    })

df_triage_curve = pd.DataFrame(triage_data)

conf_bins = np.linspace(0, 1, 11)
df_triage_curve['conf_bin'] = pd.cut(df_triage_curve['avg_conf'], bins=conf_bins)
median_cer_per_bin = df_triage_curve.groupby('conf_bin')['cer_name'].median()

threshold_conf = 0.0
for interval, med_cer in median_cer_per_bin.items():
    if pd.notna(med_cer) and med_cer <= 0.1:
        threshold_conf = interval.left
        break

print(f"Ambang batas keyakinan OCR otomatis di subset keseluruhan: {threshold_conf:.2f}")
print("Catatan: Nilai is_readable tidak disimpan ke master_df untuk mencegah kebocoran data.")
print("Ambang batas Triage sesungguhnya akan dihitung ulang secara dinamis di dalam loop validasi silang pada Notebook 3.")

master_df['avg_ocr_conf'] = master_df['filename'].map(df_triage_curve.set_index('filename')['avg_conf'])
# Baris penugasan is_readable dihapus secara sengaja.

fig, ax = plt.subplots(figsize=(8, 5))
sns.scatterplot(data=df_triage_curve, x='avg_conf', y='cer_name', alpha=0.5, ax=ax)
ax.axhline(y=0.1, color='red', linestyle='--', label='Batas Toleransi CER (0.1)')
ax.axvline(x=threshold_conf, color='green', linestyle='-', label=f'Threshold Keyakinan ({threshold_conf:.2f})')
ax.set_title('Hubungan Keyakinan Bacaan Terhadap Kesalahan Karakter')
ax.set_xlabel('Rata-rata Keyakinan OCR')
ax.set_ylabel('CER Nama Minimum')
ax.legend()
plt.tight_layout()
plt.show()

Penarikan garis ambang batas sangat transparan dan berlandaskan bukti lapangan aktual. 
Keyakinan OCR di bawah garis toleransi nyaris senantiasa membuahkan kesalahan baca
parah, menegaskan peran Triage ini sangat menjanjikan untuk operasional lini depan.

## 9. Distribusi Label Segmen dan Panjang Teks

Menyusul pemotongan segmentasi teks, melihat secara gamblang perilaku masing masing
kelas dan kecenderungannya dari segi kuantitas dan kepadatan jumlah karakter sangat
dianjurkan guna memastikan ketajaman fitur.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
colors = sns.color_palette('muted', 4)

sns.boxplot(data=df_segments, x='label', y='text_len', palette=colors, ax=axes[0])
axes[0].set_title('Distribusi Panjang Teks Berdasarkan Label')
axes[0].set_xlabel('Kelas Label')
axes[0].set_ylabel('Panjang Teks')

label_counts = df_segments['label'].value_counts()
axes[1].bar(label_counts.index, label_counts.values, color=colors, edgecolor='white')
axes[1].set_title('Persebaran Kelas Label Pengawasan Lemah')
axes[1].set_xlabel('Kelas Label')
axes[1].set_ylabel('Jumlah Muncul')
for i, count in enumerate(label_counts.values):
    axes[1].text(i, count + 10, str(count), ha='center')

plt.tight_layout()
plt.show()

Kelas nama memiliki panjang teks yang membentang lebar dibandingkan dengan atribut 
tanggal lahir yang memusat secara stabil. Karakter fitur ini sungguh menyakinkan
dan menjamin ia layak diangkat masuk ke dalam perbendaharaan model pengklasifikasi.

## 10. Normalisasi Spasial Akhir dan Penyimpanan

Seluruh variabel metrik yang beredar dalam satuan beragam akhirnya dipadatkan 
serempak antara 0 dan 1, mempersiapkan kanvas seragam sebelum pengklasifikasi
bertindak menyerap segalanya.

In [ ]:
spatial_cols = ['y_rel', 'x_rel', 'width_rel']
mm_scaler    = MinMaxScaler()
df_segments[['y_rel_scaled', 'x_rel_scaled', 'width_rel_scaled']] = \
    mm_scaler.fit_transform(df_segments[spatial_cols].fillna(0))

df_segments['line_rank_norm'] = df_segments.groupby('filename')['line_rank'].transform(
    lambda s: (s - s.min()) / max((s.max() - s.min()), 1)
)

# Menyimpan hasil
df_segments.to_csv(os.path.join(DATASET_DIR, 'ocr_segments_features.csv'), index=False)
master_df.to_csv(MASTER_PATH, index=False)

print("Rekayasa fitur telah purna dilaksanakan.")
print(f"Kumpulan data segmen tersimpan: {len(df_segments)} baris")
print(f"Kumpulan data panduan tersimpan: {len(master_df)} baris")